# Milestone 5: Predictive & Prescriptive EDA Walkthrough
## "What is likely to happen? What should we do?" — Interactive Exploration

**Purpose**: Drill into forecasts, scenarios, and recommendations using generated evidence tables. Explore revenue predictions and quantified business actions.

**Execution**: Run cell-by-cell to explore predictive models and prescriptive recommendations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Setup
OUTPUT_TABLE_DIR = Path('../outputs/tables')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}' if abs(x) < 1e5 else f'{x:.0f}')

print('Setup complete. Loading predictive/prescriptive tables...')

## Part 1: Time-Series Decomposition
**Question**: What components drive revenue variation?

In [ ]:
# Load decomposition variance
decomp = pd.read_csv(OUTPUT_TABLE_DIR / 'm5_decomposition_variance.csv')
print("Time-Series Decomposition Variance Contribution:")
display(decomp)

# Visualize variance contribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = sns.color_palette('husl', len(decomp))
ax1.bar(decomp['component'], decomp['variance']/1e12, color=colors)
ax1.set_ylabel('Variance ($T²)', fontsize=11)
ax1.set_title('Revenue Variance by Component')
ax1.tick_params(axis='x', rotation=45)
for i, (comp, var) in enumerate(zip(decomp['component'], decomp['variance']/1e12)):
    ax1.text(i, var + 0.05, f'${var:.1f}T²', ha='center', fontsize=9)

# Pie chart (% contribution)
ax2.pie(decomp['pct_total'], labels=decomp['component'], autopct='%1.1f%%', colors=colors)
ax2.set_title('Variance Contribution (%)')

plt.tight_layout()
plt.show()

print(f"\n✓ Key Insight: Trend + Seasonality + Promotional explain {decomp['pct_total'].iloc[:-1].sum():.1%} of variance")

## Part 2: Ensemble Forecasts
**Question**: What are the predicted revenue levels for the test period?

In [ ]:
# Load forecasts
forecasts = pd.read_csv(OUTPUT_TABLE_DIR / 'm5_ensemble_forecasts_base.csv')
forecasts['Date'] = pd.to_datetime(forecasts['Date'])

print(f"Forecast Period: {forecasts['Date'].min()} to {forecasts['Date'].max()}")
print(f"Total forecasted days: {len(forecasts)}")
print(f"\nForecast Summary Statistics:")
display(forecasts[['forecast_exponential', 'forecast_seasonal_naive', 'forecast_trend_seasonal', 'forecast_ensemble']].describe())

In [ ]:
# Visualize ensemble forecast vs. methods
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(forecasts['Date'], forecasts['forecast_exponential']/1e6, label='Exponential Smoothing', linewidth=1.5, alpha=0.7)
ax.plot(forecasts['Date'], forecasts['forecast_seasonal_naive']/1e6, label='Seasonal Naive', linewidth=1.5, alpha=0.7)
ax.plot(forecasts['Date'], forecasts['forecast_trend_seasonal']/1e6, label='Trend + Seasonality', linewidth=1.5, alpha=0.7)
ax.plot(forecasts['Date'], forecasts['forecast_ensemble']/1e6, label='Ensemble (Average)', linewidth=2.5, color='black', linestyle='-')

ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Revenue (Millions $)', fontsize=11)
ax.set_title('Ensemble Revenue Forecast: Three Methods + Average')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Method agreement analysis
forecasts['std_dev'] = forecasts[['forecast_exponential', 'forecast_seasonal_naive', 'forecast_trend_seasonal']].std(axis=1)
forecasts['cv'] = forecasts['std_dev'] / forecasts['forecast_ensemble']  # Coefficient of variation
print(f"\nMethod Agreement Analysis:")
print(f"  Avg Coefficient of Variation: {forecasts['cv'].mean():.3f}")
print(f"  → CV < 0.1 = high agreement (methods agree closely)")
print(f"  → CV > 0.2 = divergent (methods disagree)")
print(f"\n✓ Interpretation: Methods are {('highly' if forecasts['cv'].mean() < 0.1 else 'moderately' if forecasts['cv'].mean() < 0.2 else 'somewhat')} aligned, suggesting robust ensemble forecast")

In [ ]:
# Annual revenue projection
forecasts['month'] = forecasts['Date'].dt.to_period('M')
monthly_forecast = forecasts.groupby('month').agg({
    'forecast_ensemble': 'sum'
}).reset_index()
monthly_forecast['month'] = monthly_forecast['month'].astype(str)
monthly_forecast['month_num'] = pd.to_datetime(monthly_forecast['month']).dt.month
monthly_forecast = monthly_forecast.sort_values('month_num')

print(f"\nMonthly Revenue Forecast (Test Period):")
display(monthly_forecast[['month', 'forecast_ensemble']].head(12))

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(range(len(monthly_forecast)), monthly_forecast['forecast_ensemble']/1e9, color='steelblue', alpha=0.7)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Revenue (Billions $)', fontsize=11)
ax.set_title('Monthly Revenue Forecast (Jan 2023 - Jul 2024)')
ax.set_xticks(range(len(monthly_forecast)))
ax.set_xticklabels(monthly_forecast['month'], rotation=45, ha='right')
plt.tight_layout()
plt.show()

total_forecast = forecasts['forecast_ensemble'].sum()
print(f"\nTotal Forecasted Revenue (Test Period): ${total_forecast/1e9:.1f}B")
print(f"Average Daily Revenue: ${total_forecast/len(forecasts)/1e6:.1f}M")

## Part 3: Scenario Analysis
**Question**: What is the revenue impact of different strategies?

In [ ]:
# Load scenarios
scenarios = pd.read_csv(OUTPUT_TABLE_DIR / 'm5_scenario_analysis.csv')

# Aggregate by scenario
scenario_totals = pd.DataFrame({
    'Scenario': ['Base', 'Promotional Boost', 'Channel Optimization', 'Traffic Increase', 
                  'Inventory Optimization', 'Product Mix Shift', 'Return Reduction',
                  'Conservative', 'Moderate', 'Aggressive'],
    'Total Revenue': [
        scenarios['scenario_base'].sum(),
        scenarios['scenario_promotional_boost'].sum(),
        scenarios['scenario_channel_optimization'].sum(),
        scenarios['scenario_traffic_increase'].sum(),
        scenarios['scenario_inventory_optimization'].sum(),
        scenarios['scenario_product_mix_shift'].sum(),
        scenarios['scenario_return_reduction'].sum(),
        scenarios['scenario_conservative'].sum(),
        scenarios['scenario_moderate'].sum(),
        scenarios['scenario_aggressive'].sum()
    ]
})

# Calculate impact
base_revenue = scenario_totals['Total Revenue'].iloc[0]
scenario_totals['Impact ($B)'] = (scenario_totals['Total Revenue'] - base_revenue) / 1e9
scenario_totals['Impact (%)'] = (scenario_totals['Total Revenue'] - base_revenue) / base_revenue * 100

print("Scenario Analysis: Revenue Impact")
display(scenario_totals[['Scenario', 'Total Revenue', 'Impact ($B)', 'Impact (%)']])

In [ ]:
# Visualize scenarios
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Individual tactics
tactical_scenarios = scenario_totals[scenario_totals['Scenario'].isin([
    'Promotional Boost', 'Channel Optimization', 'Traffic Increase',
    'Inventory Optimization', 'Product Mix Shift', 'Return Reduction'
])].copy()

colors1 = ['#2ecc71' if x > 0 else '#e74c3c' for x in tactical_scenarios['Impact ($B)']]
ax1.barh(tactical_scenarios['Scenario'], tactical_scenarios['Impact ($B)'], color=colors1)
ax1.set_xlabel('Revenue Impact ($B)', fontsize=11)
ax1.set_title('Individual Scenario Impacts')
ax1.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
for i, (scenario, impact) in enumerate(zip(tactical_scenarios['Scenario'], tactical_scenarios['Impact ($B)'])):
    ax1.text(impact + 5, i, f'${impact:.1f}B (+{tactical_scenarios.iloc[i]["Impact (%)"]:.1f}%)', va='center', fontsize=9)

# Combined scenarios
combined_scenarios = scenario_totals[scenario_totals['Scenario'].isin([
    'Base', 'Conservative', 'Moderate', 'Aggressive'
])].copy()

colors2 = ['#95a5a6', '#f39c12', '#3498db', '#e74c3c']
ax2.bar(combined_scenarios['Scenario'], combined_scenarios['Total Revenue']/1e9, color=colors2)
ax2.set_ylabel('Total Revenue ($B)', fontsize=11)
ax2.set_title('Combined Scenario Revenue Projections')
for i, (scenario, revenue, impact) in enumerate(zip(combined_scenarios['Scenario'], 
                                                      combined_scenarios['Total Revenue']/1e9,
                                                      combined_scenarios['Impact (%)'])):
    label = f'${revenue:.1f}B' if i == 0 else f'${revenue:.1f}B\n(+{impact:.1f}%)'
    ax2.text(i, revenue + 0.1, label, ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\n✓ Scenario Rankings (by impact):")
ranked = scenario_totals[scenario_totals['Scenario'].isin([
    'Promotional Boost', 'Channel Optimization', 'Traffic Increase',
    'Inventory Optimization', 'Product Mix Shift', 'Return Reduction'
])].sort_values('Impact ($B)', ascending=False)
for idx, (scenario, impact) in enumerate(zip(ranked['Scenario'], ranked['Impact ($B)']), 1):
    print(f"  {idx}. {scenario}: +${impact:.1f}B")

## Part 4: Recommendations with ROI
**Question**: What are the best actions with quantified ROI?

In [ ]:
# Load recommendations
recs = pd.read_csv(OUTPUT_TABLE_DIR / 'm5_recommendations_roi.csv')

print("Five Prioritized Recommendations:")
display(recs[['recommendation_id', 'title', 'category', 'priority', 'revenue_impact', 'estimated_cost', 'roi_percent', 'timeframe_months']])

In [ ]:
# Visualize ROI
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. ROI by recommendation (sorted)
recs_sorted = recs.sort_values('roi_percent', ascending=True)
colors_roi = ['#e74c3c' if x < 100 else '#f39c12' if x < 500 else '#2ecc71' for x in recs_sorted['roi_percent']]
axes[0, 0].barh(recs_sorted['title'], recs_sorted['roi_percent'], color=colors_roi)
axes[0, 0].set_xlabel('ROI (%)', fontsize=11)
axes[0, 0].set_title('Return on Investment by Recommendation')
for i, (title, roi) in enumerate(zip(recs_sorted['title'], recs_sorted['roi_percent'])):
    axes[0, 0].text(roi + 50, i, f'{roi:.0f}%', va='center', fontsize=9)

# 2. Revenue impact
recs_rev = recs.sort_values('revenue_impact', ascending=True)
axes[0, 1].barh(recs_rev['title'], recs_rev['revenue_impact']/1e9, color='steelblue')
axes[0, 1].set_xlabel('Revenue Impact ($B)', fontsize=11)
axes[0, 1].set_title('Annual Revenue Impact by Recommendation')
for i, (title, impact) in enumerate(zip(recs_rev['title'], recs_rev['revenue_impact']/1e9)):
    axes[0, 1].text(impact + 5, i, f'${impact:.1f}B', va='center', fontsize=9)

# 3. Cost vs. Revenue (bubble chart)
axes[1, 0].scatter(recs['estimated_cost']/1e9, recs['revenue_impact']/1e9, 
                   s=500, alpha=0.6, c=recs['roi_percent'], cmap='RdYlGn')
axes[1, 0].set_xlabel('Implementation Cost ($B)', fontsize=11)
axes[1, 0].set_ylabel('Revenue Impact ($B)', fontsize=11)
axes[1, 0].set_title('Cost vs. Impact (bubble size = ROI)')
for idx, row in recs.iterrows():
    axes[1, 0].annotate(row['recommendation_id'], 
                        (row['estimated_cost']/1e9, row['revenue_impact']/1e9),
                        fontsize=10, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# 4. Implementation timeline
recs_time = recs.sort_values('timeframe_months')
priority_colors = {'Critical': '#e74c3c', 'High': '#f39c12', 'Medium': '#3498db'}
colors_priority = [priority_colors.get(p, '#95a5a6') for p in recs_time['priority']]
axes[1, 1].barh(recs_time['title'], recs_time['timeframe_months'], color=colors_priority)
axes[1, 1].set_xlabel('Implementation Timeline (Months)', fontsize=11)
axes[1, 1].set_title('Implementation Effort & Priority')
for i, (title, months, priority) in enumerate(zip(recs_time['title'], recs_time['timeframe_months'], recs_time['priority'])):
    axes[1, 1].text(months + 0.2, i, f'{int(months)}mo ({priority})', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Detailed recommendation view
print("\nDetailed Recommendation Analysis:\n")
for idx, row in recs.iterrows():
    print(f"\n{'='*80}")
    print(f"{row['recommendation_id']}: {row['title']}")
    print(f"{'='*80}")
    print(f"Category: {row['category']} | Priority: {row['priority']} | Timeline: {row['timeframe_months']} months")
    print(f"\nRevenue Impact: ${row['revenue_impact']/1e9:.1f}B")
    print(f"Implementation Cost: ${row['estimated_cost']/1e9:.1f}B")
    print(f"Net ROI: {row['roi_percent']:.0f}%")
    print(f"\nDescription: {row['description']}")
    print(f"\nEvidence: {row['evidence_basis']}")

## Part 5: Implementation Roadmap
**Question**: What's the optimal execution sequence?

In [ ]:
# Implementation phases
phases = {
    'Phase 1 (0-2 months)': ['R2: Channel Optimization', 'R5: Return Reduction'],
    'Phase 2 (2-4 months)': ['R1: Promotional Activity', 'R4: Product Mix Shift'],
    'Phase 3 (4-6 months)': ['R3: Inventory Optimization']
}

print("\nImplementation Roadmap:")
print("\nRationale:")
print("  Phase 1: Quick wins (low cost, high ROI, fast payback)")
print("  Phase 2: Medium-term initiatives (moderate complexity, high impact)")
print("  Phase 3: Strategic transformation (high complexity, highest potential)")
print()

for phase, recs_list in phases.items():
    print(f"\n{phase}:")
    for rec in recs_list:
        rec_id = rec.split(':')[0]
        rec_data = recs[recs['recommendation_id'] == rec_id].iloc[0]
        print(f"  ✓ {rec}")
        print(f"    Impact: ${rec_data['revenue_impact']/1e9:.1f}B | Cost: ${rec_data['estimated_cost']/1e9:.1f}B | ROI: {rec_data['roi_percent']:.0f}%")

## Summary: Predictive & Prescriptive Insights

**Predictive (M5):** Ensemble forecast for Jan 2023 - Jul 2024 predicts **~$15.4B revenue** using time-series decomposition of trend, seasonality, and promotional effects.

**Prescriptive (M5):** Five prioritized recommendations can drive **+$779M incremental revenue (+4.8%)** through:
1. Promotional optimization (+$230M)
2. Channel efficiency (+$43M)
3. Inventory coordination (+$184M)
4. Product mix shift (+$276M)
5. Return reduction (+$46M)

**Best Quick Wins:** R2 (Channel) and R5 (Returns) in Phase 1 → +$89M with <$25M cost → 254% ROI in 2 months

---

**Next Layer: Forecasting Models (Milestone 6 & 7)**
- Build production-ready forecasting models (ARIMA, Prophet, LightGBM)
- Validate on test period
- Generate final submission predictions (Part 3)

In [ ]:
print("\n" + "="*80)
print("MILESTONE 5 PREDICTIVE & PRESCRIPTIVE EDA COMPLETE")
print("="*80)
print("\nAll evidence tables available in: outputs/tables/m5_*.csv")
print("Formal report: docs/competition/first-round/planning/m5-predictive-prescriptive-eda-report.md")
print("\nReady for Milestone 6 (Forecasting Baseline Model)")